In [1]:
# %pip install cudf-cu12 dask-cudf-cu12 --extra-index-url=https://pypi.nvidia.com
%pip install tensorboard

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
from tqdm import tqdm
from datetime import datetime
import uuid
import csv
from torchvision.models.segmentation import deeplabv3_resnet50
from torch.utils.tensorboard import SummaryWriter
import torch.nn.functional as F

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
class LIDCSliceDataset(Dataset):
    
    def __init__(self, data_dir='lidc_processed', transform=None, preload=True):
        self.data_dir = data_dir
        self.transform = transform
        self.preload = preload
        
        self.image_dir = os.path.join(data_dir, 'images')
        self.mask_dir = os.path.join(data_dir, 'masks')
        
        self.sample_ids = [
            f.replace('.npy', '') 
            for f in os.listdir(self.image_dir) 
            if f.endswith('.npy')
        ]
        
        print(f"Found {len(self.sample_ids)} slices")
        
        if self.preload:
            print("Preloading dataset into RAM...")
            self.images = {}
            self.masks = {}
            
            for sample_id in tqdm(self.sample_ids, desc="Loading data"):
                image = np.load(os.path.join(self.image_dir, f'{sample_id}.npy'))
                mask = np.load(os.path.join(self.mask_dir, f'{sample_id}.npy'))
                
                if len(image.shape) == 2:
                    image = np.stack([image, image, image], axis=0)
                else:
                    image = np.transpose(image, (2, 0, 1))
                    if image.shape[0] == 1:
                        image = np.repeat(image, 3, axis=0)
                
                if len(mask.shape) == 2:
                    mask = mask
                else:
                    mask = mask[..., 0]
                
                self.images[sample_id] = torch.from_numpy(image).float()
                self.masks[sample_id] = torch.from_numpy(mask).long()
            
            print("Dataset preloaded into RAM")
    
    def __len__(self):
        return len(self.sample_ids)
    
    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        
        if self.preload:
            image = self.images[sample_id]
            mask = self.masks[sample_id]
        else:
            image = np.load(os.path.join(self.image_dir, f'{sample_id}.npy'))
            mask = np.load(os.path.join(self.mask_dir, f'{sample_id}.npy'))
            
            if len(image.shape) == 2:
                image = np.stack([image, image, image], axis=0)
            else:
                image = np.transpose(image, (2, 0, 1))
                if image.shape[0] == 1:
                    image = np.repeat(image, 3, axis=0)
            
            if len(mask.shape) == 2:
                mask = mask
            else:
                mask = mask[..., 0]
            
            image = torch.from_numpy(image).float()
            mask = torch.from_numpy(mask).long()
        
        return {
            'image': image,
            'mask': mask,
            'id': sample_id
        }

In [3]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    
    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        pred = pred.contiguous().view(-1)
        target = target.contiguous().view(-1)
        
        intersection = (pred * target).sum()
        dice = (2. * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth)
        
        return 1 - dice

In [4]:
def calculate_dice_score(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    
    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)
    
    intersection = (pred * target).sum()
    dice = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)
    
    return dice.item()

In [5]:
class DeepLabV3Segmentation:
    
    def __init__(self, num_classes=2, device='cuda', learning_rate=1e-4, 
                 log_dir_prefix='runs/deeplab'): 
        
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        self.num_classes = num_classes
        
        self.model = deeplabv3_resnet50(weights='DEFAULT')
        self.model.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
        self.model.aux_classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
        self.model = self.model.to(self.device)
        
        self.criterion = DiceLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)
        
        os.makedirs('checkpoints', exist_ok=True)
        os.makedirs('metrics', exist_ok=True)
        
        # --- Generates unique log_dir ---
        datetime_str = datetime.now().strftime("%Y%m%d_%H%M%S")
        unique_hash = uuid.uuid4().hex[:8]
        log_dir = f"{log_dir_prefix}_{datetime_str}_{unique_hash}"
        # ---
        
        self.writer = SummaryWriter(log_dir)
        print(f"TensorBoard logs will be saved to: {log_dir}")

    def train_epoch(self, dataloader, epoch):
        self.model.train()
        epoch_loss = 0.0
        epoch_dice = 0.0
        num_batches = 0
        
        progress_bar = tqdm(dataloader, desc=f'Training Epoch {epoch}')
        
        for i, batch in enumerate(progress_bar): 
            images = batch['image'].to(self.device)
            masks = batch['mask'].to(self.device)
            
            masks_float = masks.float().unsqueeze(1) 
            
            self.optimizer.zero_grad()
            
            outputs = self.model(images)
            logits = outputs['out']
            
            logits_class1 = logits[:, 1:2, :, :]
            
            loss = self.criterion(logits_class1, masks_float)
            
            loss.backward()
            self.optimizer.step()
            
            dice = calculate_dice_score(logits_class1, masks_float)
            
            epoch_loss += loss.item()
            epoch_dice += dice
            num_batches += 1
            
            global_step = (epoch - 1) * len(dataloader) + i
            self.writer.add_scalar('Loss/train_batch', loss.item(), global_step)
            self.writer.add_scalar('DICE/train_batch', dice, global_step)
            
            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'dice': f'{dice:.4f}'
            })
            
        return epoch_loss / num_batches, epoch_dice / num_batches

    def validate(self, dataloader, epoch):
        self.model.eval()
        val_loss = 0.0
        val_dice = 0.0
        num_batches = 0
        
        with torch.no_grad():
            progress_bar = tqdm(dataloader, desc=f'Validation Epoch {epoch}')
            
            for i, batch in enumerate(progress_bar): 
                images = batch['image'].to(self.device)
                masks = batch['mask'].to(self.device)
                
                masks_float = masks.float().unsqueeze(1)
                
                outputs = self.model(images)
                logits = outputs['out']
                
                logits_class1 = logits[:, 1:2, :, :]
                
                loss = self.criterion(logits_class1, masks_float)
                dice = calculate_dice_score(logits_class1, masks_float)
                
                val_loss += loss.item()
                val_dice += dice
                num_batches += 1
                
                progress_bar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'dice': f'{dice:.4f}'
                })
                
                if i == 0: 
                    preds = torch.sigmoid(logits_class1) 
                    self.writer.add_images('Validation/Input_Images', images, epoch)
                    self.writer.add_images('Validation/Target_Masks', masks_float, epoch)
                    self.writer.add_images('Validation/Model_Predictions', preds, epoch)

        return val_loss / num_batches, val_dice / num_batches

    def save_checkpoint(self, epoch, train_loss, train_dice, val_loss, val_dice):
        checkpoint_path = os.path.join('checkpoints', f'deeplabv3_luna_epoch_{epoch}.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'train_loss': train_loss,
            'train_dice': train_dice,
            'val_loss': val_loss,
            'val_dice': val_dice
        }, checkpoint_path)
        
        return checkpoint_path

    def save_metrics(self, epoch, train_loss, train_dice, val_loss, val_dice, checkpoint_path):
        csv_path = os.path.join('metrics', f'epoch_{epoch}_metrics.csv')
        
        with open(csv_path, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Metric', 'Value'])
            writer.writerow(['Epoch', epoch])
            writer.writerow(['Train Loss', train_loss])
            writer.writerow(['Train DICE', train_dice])
            writer.writerow(['Validation Loss', val_loss])
            writer.writerow(['Validation DICE', val_dice])
            writer.writerow(['Checkpoint Path', checkpoint_path])
            writer.writerow(['Model', 'DeepLabV3-ResNet50'])
            writer.writerow(['Loss Function', 'Dice Loss'])
            writer.writerow(['Optimizer', 'Adam'])
        
        return csv_path

#     def train(self, train_loader, val_loader, num_epochs):
#         print(f"Training on device: {self.device}")
#         print(f"Model: DeepLabV3-ResNet50")
#         print(f"Loss: Dice Loss")
#         print(f"Optimizer: Adam")
#         print(f"Number of epochs: {num_epochs}")
#         print("-" * 50)
        
#         try:
#             sample_batch = next(iter(train_loader))
#             sample_images = sample_batch['image'].to(self.device)
#             self.writer.add_graph(self.model, sample_images)
#             print("Model graph added to TensorBoard.")
#         except Exception as e:
#             print(f"Could not add model graph to TensorBoard: {e}")
        
#         for epoch in range(1, num_epochs + 1):
#             print(f"\nEpoch {epoch}/{num_epochs}")
            
#             train_loss, train_dice = self.train_epoch(train_loader, epoch)
#             val_loss, val_dice = self.validate(val_loader, epoch)
            
#             print(f"\nEpoch {epoch} Summary:")
#             print(f"Train Loss: {train_loss:.4f}, Train DICE: {train_dice:.4f}")
#             print(f"Val Loss: {val_loss:.4f}, Val DICE: {val_dice:.4f}")
            
#             self.writer.add_scalar('Loss/train_epoch', train_loss, epoch)
#             self.writer.add_scalar('DICE/train_epoch', train_dice, epoch)
#             self.writer.add_scalar('Loss/validation_epoch', val_loss, epoch)
#             self.writer.add_scalar('DICE/validation_epoch', val_dice, epoch)
#             self.writer.add_scalar('Params/learning_rate', self.optimizer.param_groups[0]['lr'], epoch)
            
#             checkpoint_path = self.save_checkpoint(epoch, train_loss, train_dice, val_loss, val_dice)
#             csv_path = self.save_metrics(epoch, train_loss, train_dice, val_loss, val_dice, checkpoint_path)
            
#             print(f"Saved checkpoint: {checkpoint_path}")
#             print(f"Saved metrics: {csv_path}")
#             print("-" * 50)
            
#         self.writer.close()
#         print("Training finished. TensorBoard logs saved.")
        
    def train(self, train_loader, val_loader, num_epochs):
        print(f"Training on device: {self.device}")
        print(f"Model: DeepLabV3-ResNet50")
        print(f"Loss: Dice Loss")
        print(f"Optimizer: Adam")
        print(f"Number of epochs: {num_epochs}")
        print("-" * 50)
        
        try:
            sample_batch = next(iter(train_loader))
            sample_images = sample_batch['image'].to(self.device)
            
            class ModelWrapper(nn.Module):
                def __init__(self, model):
                    super(ModelWrapper, self).__init__()
                    self.model = model
                
                def forward(self, x):
                    return self.model(x)['out']

            self.writer.add_graph(ModelWrapper(self.model), sample_images)
            print("Model graph added to TensorBoard.")
        except Exception as e:
            print(f"Could not add model graph to TensorBoard: {e}")
        
        for epoch in range(1, num_epochs + 1):
            print(f"\nEpoch {epoch}/{num_epochs}")
            
            train_loss, train_dice = self.train_epoch(train_loader, epoch)
            val_loss, val_dice = self.validate(val_loader, epoch)
            
            print(f"\nEpoch {epoch} Summary:")
            print(f"Train Loss: {train_loss:.4f}, Train DICE: {train_dice:.4f}")
            print(f"Val Loss: {val_loss:.4f}, Val DICE: {val_dice:.4f}")
            
            self.writer.add_scalar('Loss/train_epoch', train_loss, epoch)
            self.writer.add_scalar('DICE/train_epoch', train_dice, epoch)
            self.writer.add_scalar('Loss/validation_epoch', val_loss, epoch)
            self.writer.add_scalar('DICE/validation_epoch', val_dice, epoch)
            self.writer.add_scalar('Params/learning_rate', self.optimizer.param_groups[0]['lr'], epoch)
            
            checkpoint_path = self.save_checkpoint(epoch, train_loss, train_dice, val_loss, val_dice)
            csv_path = self.save_metrics(epoch, train_loss, train_dice, val_loss, val_dice, checkpoint_path)
            
            print(f"Saved checkpoint: {checkpoint_path}")
            print(f"Saved metrics: {csv_path}")
            print("-" * 50)
            
        self.writer.close()
        print("Training finished. TensorBoard logs saved.")

In [ ]:
dataset = LIDCSliceDataset('luna_lidc_processed', preload=False)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

segmentation = DeepLabV3Segmentation(num_classes=2, device='cuda', learning_rate=1e-4)

segmentation.train(train_loader, val_loader, num_epochs=50)

Found 1576 slices
Train samples: 1260
Validation samples: 316


Downloading: "https://download.pytorch.org/models/deeplabv3_resnet50_coco-cd0a2569.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_resnet50_coco-cd0a2569.pth
100%|██████████| 161M/161M [00:07<00:00, 23.8MB/s] 


TensorBoard logs will be saved to: runs/deeplab_20260107_004411_37fb4e44
Training on device: cuda
Model: DeepLabV3-ResNet50
Loss: Dice Loss
Optimizer: Adam
Number of epochs: 50
--------------------------------------------------
Model graph added to TensorBoard.

Epoch 1/50


Training Epoch 1:  52%|█████▏    | 82/158 [00:53<00:47,  1.60it/s, loss=0.9955, dice=0.0140]